<a href="https://colab.research.google.com/github/simecek/dspracticum2026/blob/main/lesson02/04_fastai.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 4: The same with fastai

1. Python basics & the training loop
2. Dense neural network on FashionMNIST
3. Convolutional neural network (CNN) on FashionMNIST
4. **The same with fastai** ← *you are here*
5. Fine-tuning a pretrained model

You already know the data (FashionMNIST) and the model (our CNN from notebook 3). So this notebook is only about **what [fastai](https://docs.fast.ai/) brings**. fastai is a library built on top of PyTorch that:
- replaces our 25-line training loop with **one line**
- **finds a good learning rate** for us, instead of guessing
- trains with smarter defaults (**Adam** optimizer, **one-cycle** schedule)
- shows us where the model fails, including the images it got **most wrong**

**Before you start:** *Runtime → Change runtime type → T4 GPU*.

In [ ]:
from fastai.vision.all import *

This one line imports everything we need: fastai, but also `torch`, `nn`, `plt`... (Importing with `*` is usually frowned upon, but it is the standard way of using fastai.)

---
## 1. Data: images in folders

fastai, like most real-world projects, expects images as **files, one folder per class**. It is the same format you will use for your own datasets:

```
fashion_mnist/
├── train/
│   ├── Ankle boot/     6,000 images
│   ├── Bag/            6,000 images
│   └── ...
└── test/
    ├── Ankle boot/     1,000 images
    └── ...
```

The cell below saves FashionMNIST in this format. Just run it.

In [ ]:
# @title Save FashionMNIST as image files (just run this cell, takes less than a minute)
from torchvision import datasets

path = Path("fashion_mnist")
if not path.exists():
    for split, train in [("train", True), ("test", False)]:
        dataset = datasets.FashionMNIST(root="data", train=train, download=True)
        for i, (image, label) in enumerate(dataset):
            folder = path / split / dataset.classes[label].replace("/", "-")
            folder.mkdir(parents=True, exist_ok=True)
            image.save(folder / f"{i}.png")
print("done")

In [ ]:
print((path / "train").ls())
print("images of bags:", len((path / "train" / "Bag").ls()))

**One line** creates the data loaders from the folders:
- the **folder names become the labels**
- `valid="test"`: fastai calls the images used to check the model the **validation set**
- `img_cls=PILImageBW`: our images are black & white (the default would be color)

In [ ]:
dls = ImageDataLoaders.from_folder(path, train="train", valid="test", img_cls=PILImageBW, bs=64)
dls.show_batch(max_n=12, figsize=(10, 6), cmap="gray")

### Look inside: it's the same data as before

In [ ]:
print("classes:", dls.vocab)

images, labels = dls.one_batch()
print("images:", images.shape)     # [64, 1, 28, 28], exactly as in notebooks 2 and 3
print("labels:", labels.shape)

---
## 2. The Learner

Our CNN from notebook 3, **copied without any change**. fastai works with any PyTorch model:

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2)
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.25)

    def forward(self, x):
        x = self.pool(self.relu(self.conv1(x)))
        x = self.pool(self.relu(self.conv2(x)))
        x = self.flatten(x)
        x = self.dropout(self.relu(self.fc1(x)))
        x = self.fc2(x)
        return x

A **`Learner`** bundles everything needed for training: the data, the model, the loss and the metric we want to watch. (`CrossEntropyLossFlat` is fastai's version of the same cross-entropy loss.)

In [ ]:
learn = Learner(dls, CNN(), loss_func=CrossEntropyLossFlat(), metrics=accuracy)

### Look inside: `learn.summary()`

Remember our cells tracing the shapes and counting parameters? fastai does both in one call. (Layers we use twice, like `pool` and `relu`, are listed only once.)

In [ ]:
learn.summary()

---
## 3. Finding the learning rate

In notebook 1 we saw that a learning rate that is too small learns slowly, and one that is too big explodes. So far we simply **guessed** `lr=0.1`.

`lr_find()` runs a short experiment instead: it trains on a few batches while increasing the learning rate from tiny to huge, and plots the loss. (Afterwards, the model is reset, so nothing is spoiled.)

In [ ]:
suggestion = learn.lr_find()
suggestion

How to read the plot, from left to right:
- **flat**: the learning rate is too small, nothing happens
- **going down**: the model is learning. Good values are here, in the steepest part.
- **shooting up**: too big, the training explodes

The orange dot (*valley*) is fastai's suggestion: a safe, but rather cautious choice. Any value in the steep downhill part works, and bigger steps mean faster learning. We will take **0.005** (written `5e-3`), to the right of the dot, where the loss is still going down steeply.

---
## 4. Training in one line

Our whole training loop from notebooks 2 and 3 (forward, loss, backward, update, reset, plus computing accuracy after each epoch) is now a single line:

In [ ]:
learn.fit_one_cycle(5, 5e-3)

Better than notebook 3 (~89%) with the **same model, same data, same number of epochs**. Why? Two tricks that fastai uses by default:

1. **Adam** instead of plain SGD: an optimizer that adapts the step size for each parameter separately.
2. **One-cycle** learning rate schedule: the learning rate starts low, warms up to the maximum and then cools down to almost zero, for careful final steps. Let's look at it (the right plot shows *momentum*, another setting that fastai changes in the opposite direction; you can ignore it for now):

In [ ]:
learn.recorder.plot_sched()

And the loss during training, for free:

In [ ]:
learn.recorder.plot_loss()
plt.show()

### Look inside: it's still PyTorch

fastai is not magic. `learn.model` is our CNN, and under the hood fastai runs the same 5-step training loop. We can look at the learned kernels exactly as in notebook 3:

In [ ]:
print(type(learn.model))
print("kernels of the first layer:", learn.model.conv1.weight.shape)
print("optimizer:", learn.opt_func.__name__)

---
## 5. Where does the model fail?

In notebook 3 we needed a dozen lines for the confusion matrix. With fastai:

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix(figsize=(7, 7))

Or as a list of the most frequent mistakes:

In [ ]:
for true, predicted, count in interp.most_confused(min_val=40):
    print(f"{true:12s} predicted as {predicted:12s} {count} times")

### Look inside: the worst mistakes

`plot_top_losses` shows the images with the **highest loss**, where the model was **confident and wrong**. The title says: *predicted / true label / loss / probability of the predicted class*.

This is one of the most useful tools in practice. Such images are often ambiguous, and sometimes the **label is simply wrong**. (With your own datasets, this is how you find mislabeled images!)

In [ ]:
interp.plot_top_losses(9, figsize=(13, 13), cmap="gray")

**Try it:** Do you agree with the labels? Would you have guessed better than the model?

---
## 6. Predicting a single image

`learn.predict` takes an image and returns the predicted class, its index and the probabilities of all classes:

In [ ]:
file = (path / "test" / "Sneaker").ls()[0]
image = PILImageBW.create(file)
image.show(figsize=(3, 3), cmap="gray")
plt.show()

prediction, index, probabilities = learn.predict(image)
print("prediction:", prediction)
for name, p in zip(dls.vocab, probabilities):
    print(f"{name:12s} {p:.3f}")

---
## 7. Summary

| | PyTorch (notebooks 2 and 3) | fastai |
|---|---|---|
| data | `DataLoader(...)` | `ImageDataLoaders.from_folder(...)` |
| look at data | a loop with `plt.subplots` | `dls.show_batch()` |
| shapes & parameters | our own cells | `learn.summary()` |
| learning rate | a guess | `learn.lr_find()` |
| training | a 25-line loop | `learn.fit_one_cycle(5, lr)` |
| optimizer | SGD, fixed learning rate | Adam + one-cycle schedule |
| loss curves | our own plotting code | `learn.recorder.plot_loss()` |
| confusion matrix | a dozen lines | `interp.plot_confusion_matrix()` |
| worst mistakes | - | `interp.plot_top_losses()` |

**The model is the same PyTorch model.** fastai takes care of the boring parts and adds good defaults.

### Exercises
1. Train with the *valley* suggestion instead of `5e-3`, and then with `5e-2` (create a new `Learner` first, otherwise you continue training the old model). Which works best?
2. Replace `learn.fit_one_cycle(5, lr)` with `learn.fit(5, lr)`, which uses a constant learning rate. Is one-cycle better?
3. Copy the `DenseNet` from notebook 2 and train it with fastai. How close does it get to the CNN?
4. Train the CNN for 15 epochs and look at `learn.recorder.plot_loss()`. Does the validation loss keep going down?

**Next:** in notebook 5 we stop training from scratch. We take a network already trained on millions of photos and **fine-tune** it on a new dataset in a few minutes.